# Load Dataset

In [90]:
from google.colab import drive
drive.mount('/content/drive')
# drive.mount('/content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/News Articles')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [91]:
import os
import numpy as np
import chardet

file_path = '/content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/News Articles'

texts = []
labels = []

for genre in os.listdir(file_path):
    genre_path = os.path.join(file_path, genre)
    if os.path.isdir(genre_path):
        for article_file in os.listdir(genre_path):
            if article_file.endswith('.txt'):
                full_path = os.path.join(genre_path, article_file)

                with open(full_path, 'rb') as f:
                    raw_data = f.read()
                    result = chardet.detect(raw_data)
                    encoding = result['encoding']

                with open(full_path, 'r', encoding=encoding, errors='replace') as f:
                    text = f.read()
                    texts.append(text)
                    labels.append(genre)

print(f"Total number of articles loaded: {len(texts)}")
print(f"Number of unique genres: {len(set(labels))}")
print("\nGenre distribution:")
for genre in set(labels):
    print(f"{genre}: {labels.count(genre)} articles")

Total number of articles loaded: 2225
Number of unique genres: 5

Genre distribution:
entertainment: 386 articles
politics: 417 articles
tech: 401 articles
business: 510 articles
sport: 511 articles


# Process Data

In [92]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 500

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len)

print("\nVocabulary size:", len(tokenizer.word_index))
print("Shape of data tensor:", padded_sequences.shape)


Vocabulary size: 32470
Shape of data tensor: (2225, 500)


## Split data

In [93]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

x_train, x_test, y_train, y_test = train_test_split(
    padded_sequences,
    encoded_labels,
    test_size=0.2,
    random_state=42
)

print("\nTraining set shape:", x_train.shape)
print("Testing set shape:", x_test.shape)


Training set shape: (1780, 500)
Testing set shape: (445, 500)


# Create model

## GRU Model

In [106]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Dense, Dropout, LSTM, Bidirectional, Input, SimpleRNN, GRU
)
from tensorflow.keras.models import Model, Sequential

def create_gru_model(max_words, num_classes):
    model = Sequential([
        Embedding(max_words, 100),
        GRU(64, return_sequences=True),
        Dropout(0.3),
        GRU(32),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

## LSTM Model

In [107]:
def create_lstm_model(max_words, num_classes):
    model = Sequential([
        Embedding(max_words, 100),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

## Bidirectional LSTM

In [108]:
def create_bidirectional_lstm_model(max_words, num_classes):
    model = Sequential([
        Embedding(max_words, 100),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(32)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train model

In [109]:
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report

models = {
    "GRU": create_gru_model(max_words, len(set(labels))),
    "LSTM": create_lstm_model(max_words, len(set(labels))),
    "BLSTM": create_bidirectional_lstm_model(max_words, len(set(labels))),
}

def train_and_evaluate_model(model, x_train, y_train, x_test, y_test, model_name):
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        batch_size=16,
        epochs=10,
        callbacks=[early_stopping]
    )

    # Evaluate the model
    loss, accuracy = model.evaluate(x_test, y_test)
    print(f"\n{model_name} Test accuracy: {accuracy:.4f}")

    # Generate detailed classification report
    y_pred = np.argmax(model.predict(x_test), axis=1)
    print(f"\n{model_name} Classification Report:")
    print(classification_report(y_test, y_pred,
                              target_names=label_encoder.classes_))

    return history

histories = {}

# Train and evaluate each model
for model_name, model in models.items():
    print(f"\nTraining {model_name} model")
    histories[model_name] = train_and_evaluate_model(
        model, x_train, y_train, x_test, y_test, model_name
    )



Training GRU model
Epoch 1/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 56s 563ms/step - accuracy: 0.2246 - loss: 1.6049 - val_accuracy: 0.3455 - val_loss: 1.5693
Epoch 2/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 85s 603ms/step - accuracy: 0.4549 - loss: 1.3666 - val_accuracy: 0.4017 - val_loss: 1.3103
Epoch 3/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 46s 516ms/step - accuracy: 0.6623 - loss: 0.8467 - val_accuracy: 0.6376 - val_loss: 0.9442
Epoch 4/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 82s 525ms/step - accuracy: 0.8934 - loss: 0.3118 - val_accuracy: 0.6685 - val_loss: 1.0981
Epoch 5/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 82s 522ms/step - accuracy: 0.9774 - loss: 0.0970 - val_accuracy: 0.6742 - val_loss: 1.1872
Epoch 6/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 50s 559ms/step - accuracy: 0.9964 - loss: 0.0277 - val_accuracy: 0.6910 - val_loss: 1.3605
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.6627 - loss: 0.9181

GRU Test accuracy: 0.6742
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 152ms/step

GRU Classification Report:
               precision    recall 

# Save Model and configuration

In [116]:
import json

save_dir = '/content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/saved_models'
os.makedirs(save_dir, exist_ok=True)

for model_name, model in models.items():
    model_path = os.path.join(save_dir, f'{model_name}_model.keras')
    model.save(model_path)
    print(f"Saved {model_name} model to {model_path}")

Saved GRU model to /content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/saved_models/GRU_model.keras
Saved LSTM model to /content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/saved_models/LSTM_model.keras
Saved BLSTM model to /content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/saved_models/BLSTM_model.keras


# Load model and configuration

In [117]:
loaded_models = {}
for model_name in models.keys():
    model_path = os.path.join(save_dir, f'{model_name}_model.keras')
    loaded_models[model_name] = load_model(model_path)
    print(f"Loaded {model_name} model from {model_path}")

Loaded GRU model from /content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/saved_models/GRU_model.keras
Loaded LSTM model from /content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/saved_models/LSTM_model.keras
Loaded BLSTM model from /content/drive/MyDrive/ITCI5_(2024-2025)/Artificial Intelligence/Project/saved_models/BLSTM_model.keras


# Test input

In [135]:
def predict_with_models(text, models_dict=loaded_models):
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_len)

    for model_name, model in models_dict.items():
        prediction = model.predict(padded)
        predicted_genre = label_encoder.classes_[prediction.argmax()]
        probabilities = dict(zip(label_encoder.classes_, prediction[0]))

        print(f"\n{model_name} Model:")
        print(f"Predicted genre: {predicted_genre}")
        print("Probabilities:")
        for genre, prob in probabilities.items():
            print(f"{genre}: {prob:.4f}")

text = """
The “Find Song by Lyrics” (or partial lyrics)  tool can help you figure it out and solve your earworm. It’s simple—no artist name required. Just type the few lyrics you know, and once you’re finished entering them, our tool will help identify potential song matches. Don’t worry, you don’t need perfect lyrics to use this tool.
"""

predict_with_models(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step

GRU Model:
Predicted genre: politics
Probabilities:
business: 0.0596
entertainment: 0.0514
politics: 0.4851
sport: 0.1677
tech: 0.2363
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step

LSTM Model:
Predicted genre: tech
Probabilities:
business: 0.0010
entertainment: 0.1284
politics: 0.0016
sport: 0.0012
tech: 0.8678
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step

BLSTM Model:
Predicted genre: entertainment
Probabilities:
business: 0.0122
entertainment: 0.9682
politics: 0.0012
sport: 0.0109
tech: 0.0075
